In [2]:
import numpy as np 
import pandas as pd 
orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")

In [3]:
master = pd.merge(orders,customers,on="customer_id")
master = pd.merge(master,order_items,on="order_id")
master = pd.merge(master, products,on="product_id")

In [4]:
master.groupby("customer_id")["price"].sum().sort_values(ascending=False)

customer_id
1617b1357756262bfa56ab541c47bc16    13440.00
ec5b2ba62e574342386871631fafd3fc     7160.00
c6e2731c5b391845f6800c97401a43a9     6735.00
f48d464a0baaea338cb25f816991ab1f     6729.00
3fd6777bbce08a352fddd04e4a7cc8f6     6499.00
                                      ...   
d2c63ad286e3ca9dd69218008d61ff81        2.90
184e8e8e48937145eb96c721ef1f0747        2.29
a790343ca6f3fee08112d678b43aa7c5        2.20
161b6d415e8b3413c6609c70cf405b5a        0.85
9f9d249355f63c5c1216a82b802452c1        0.85
Name: price, Length: 98666, dtype: float64

In [6]:
order_count = master.groupby("customer_id")["order_id"].nunique()
single_order = order_count[order_count == 1]
print(f"Number of customers with only one order: {len(single_order)}")

Number of customers with only one order: 98666


In [15]:
master["Month"] = pd.to_datetime(master["order_purchase_timestamp"]).dt.to_period("M").astype(str)

monthly_product_sales = ( master.groupby(["product_id", "Month"], as_index=False)["price"]
    .sum()
    .rename(columns={"price": "Sales"})
)
monthly_product_sales.head()

,product_id,Month,Sales
0,00066f42aeeb9f3007548bb9d3f33c38,2018-05,101.65
1,00088930e925c41fd95ebfe695fd2655,2017-12,129.90
2,0009406fd7479715e4bef61dd91f2462,2017-12,229.00
3,000b8f95fcb9e0096488278317764d19,2018-08,117.80
4,000d9be29b5207b54e86aa1b1ac54872,2018-04,199.00


In [19]:
monthly_product_sales.columns
melted = monthly_product_sales.melt(
    id_vars="product_id",
    var_name="Month",
    value_name="Saless"
)

melted.head()

,product_id,Month,Saless
0,00066f42aeeb9f3007548bb9d3f33c38,Month,2018-05
1,00088930e925c41fd95ebfe695fd2655,Month,2017-12
2,0009406fd7479715e4bef61dd91f2462,Month,2017-12
3,000b8f95fcb9e0096488278317764d19,Month,2018-08
4,000d9be29b5207b54e86aa1b1ac54872,Month,2018-04


In [ ]:
master["customer_city"].str.upper()

master["order_purchase_timestamp"] = pd.to_datetime( master["order_purchase_timestamp"])
master["Year"] = master["order_purchase_timestamp"].dt.year.head()
master["Month"] = master["order_purchase_timestamp"].dt.month
print (master[["order_purchase_timestamp", "Year", "Month"]].head())


  order_purchase_timestamp    Year  Month
0      2017-10-02 10:56:33  2017.0     10
1      2018-07-24 20:41:37  2018.0      7
2      2018-08-08 08:38:49  2018.0      8
3      2017-11-18 19:28:06  2017.0     11
4      2018-02-13 21:18:39  2018.0      2


In [27]:
### Bonus tasks:

master["delivered"] = pd.to_datetime(master["order_delivered_customer_date"])

master["purchase"] = pd.to_datetime(master["order_purchase_timestamp"])

master["delivery_days"] = (master["delivered"] -master["purchase"]).dt.days
master["delivery_days"].mean()

np.float64(12.007722603361284)

In [28]:
master.groupby("customer_city")["delivery_days"].mean().sort_values(ascending=False)

customer_city
novo brasil                 148.00
capinzal do norte           109.00
adhemar de barros            97.00
santa cruz de goias          86.00
arace                        85.75
                             ...  
porto dos gauchos              NaN
santo antonio de goias         NaN
sao fernando                   NaN
sao francisco do humaita       NaN
sao joao do itaperiu           NaN
Name: delivery_days, Length: 4110, dtype: float64